<a href="https://colab.research.google.com/github/zacw-243L/Oracle-Engine-to-estimate-Unifrom-amount/blob/main/Oracle%20Engine%20to%20estimate%20Unifrom%20amount.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Oracle Engine

## Part 1: Prepare the uniform data

This section prepares the uniform collection file for the forecasting system.
**Just click the ▶ Run button beside each step, from 1 to 4.** Wait for each step to finish before moving on.

**Before starting:** Place the sample CSV named **`PCG MPC Uniform Collection-XXXXXXXXXXXXXXXXXXXXXXXX.csv`** in **My Drive** on Google Drive.
You can leave the filename as it is.

**When finished:** The prepared file will appear in **/content/oracle_engine_sanitizer** as **`Cleaned Data.csv`**. Your original file will remain unchanged.



In [ ]:

# @title 1. Connect Google Drive { display-mode: "form", single-column: true }
# @markdown Click **Run** to connect Google Drive and locate the collection file.

from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

_drive_folder = Path('/content/drive/MyDrive')

_source_on_drive = (
    _drive_folder /
    'PCG MPC Uniform Collection-6ab61ef0a70268fced1f06aa.csv'
)

_local_folder = Path('/content/oracle_engine_sanitizer')
_local_folder.mkdir(parents=True, exist_ok=True)

if not _source_on_drive.is_file():
    raise FileNotFoundError(
        'Collection file not found in My Drive.'
    )

# Remove an old raw copy from earlier notebook versions.
_old_raw = _local_folder / 'raw_input.csv'
_old_raw.unlink(missing_ok=True)

print('Google Drive connected.')
print('Collection file located.')
print('Please run Step 2.')


In [ ]:

# @title 2. Remove personal details { display-mode: "form", single-column: true }
# @markdown Click **Run** to prepare the uniform collection data.

import csv

_working_columns = [
    'Timestamp',
    'Intake',
    'No.4 Shirt (Size, Qty)',
    'No.4 Trousers (Size, Qty)',
    'PCG No.4 Utility Belt (Size, Qty)',
    'PCG No.4 Tactical Belt (Size, Qty)',
    'Fire & Cut Resistant Tactical Gloves (Size, Qty)',
    'Insignia: Deckhand (Qty)',
    'No.4 ERT Badge (Qty)',
    'Duffle Bag (Qty)',
    'Utility Bag (Qty)'
]

_working_path = _local_folder / 'working_data.csv'


def _sanitize_source():

    temp_path = _local_folder / 'working_data.tmp'
    temp_path.unlink(missing_ok=True)

    # Prevent reuse of an outdated working file.
    _working_path.unlink(missing_ok=True)

    count = 0
    excluded = 0

    try:
        with (
            _source_on_drive.open(
                'r', encoding='utf-8-sig', newline=''
            ) as source,
            temp_path.open(
                'w', encoding='utf-8-sig', newline=''
            ) as destination
        ):
            reader = csv.reader(source)

            # Locate the FormSG response header.
            for header_line, row in enumerate(reader, start=1):
                if {'Response ID', 'Timestamp', 'Intake'}.issubset(row):
                    columns = row
                    break
            else:
                raise ValueError('FormSG response header not found.')

            if len(columns) != len(set(columns)):
                raise ValueError('Duplicate source columns found.')

            missing = [
                c for c in _working_columns if c not in columns
            ]

            if missing:
                raise ValueError(
                    'Missing required columns: ' + ', '.join(missing)
                )

            writer = csv.DictWriter(
                destination, fieldnames=_working_columns
            )
            writer.writeheader()

            for line_number, values in enumerate(
                reader, start=header_line + 1
            ):
                if not any(v.strip() for v in values):
                    continue

                if len(values) != len(columns):
                    raise ValueError(
                        f'Invalid field count at row {line_number}.'
                    )

                response = dict(zip(columns, values))

                if response.get(
                    'Download Status', 'Success'
                ).strip().lower() != 'success':
                    excluded += 1
                    continue

                # Restrict this demonstration to fictional records.
                if not response.get(
                    'Name', ''
                ).strip().startswith('SYNTHETIC PERSON '):
                    raise ValueError(
                        'Only synthetic demonstration data is permitted.'
                    )

                # Allowlist: identifying columns never reach the output.
                cleaned = {
                    c: response[c].strip()
                    for c in _working_columns
                }

                if not cleaned['Timestamp'] or not cleaned['Intake']:
                    raise ValueError(
                        f'Missing date or intake at row {line_number}.'
                    )

                writer.writerow(cleaned)
                count += 1

        if count == 0:
            raise ValueError('No valid responses found.')

        temp_path.replace(_working_path)

        return count, excluded

    finally:
        temp_path.unlink(missing_ok=True)


_count, _excluded = _sanitize_source()

print(f'{_count:,} responses prepared.')
print(f'{_excluded:,} unsuccessful responses skipped.')
print('Identifying fields excluded.')
print('Please run Step 3.')


In [ ]:
# @title 3. Clean and save the uniform data { display-mode: "form", single-column: true }
# @markdown Click **▶ Run** to prepare the final file. It will be saved to **My Drive** as **Cleaned Data.csv**. Running this step again replaces the previous Cleaned Data.csv.
import csv
import re
import shutil
from datetime import datetime
from pathlib import Path

if '_working_path' not in globals() or not _working_path.is_file():
    raise RuntimeError('Please run Step 2 first.')

# The final 9 columns match the forecasting example exactly.
_clean_columns = [
    'Timestamp', 'Intake',
    'No.4 Shirt (Size, Qty)', 'No.4 Trousers (Size, Qty)',
    'PCG No.4 Utility Belt (Size, Qty)',
    'PCG No.4 Tactical Belt (Size, Qty)',
    'Fire & Cut Resistant Tactical Gloves (Size, Qty)',
    'Insignia: Deckhand (Qty)', 'No.4 ERT Badge (Qty)'
]

with _working_path.open('r', encoding='utf-8-sig', newline='') as file:
    _reader = csv.DictReader(file)
    if not _reader.fieldnames or any(col not in _reader.fieldnames for col in _clean_columns):
        raise ValueError('The working data is missing required uniform fields. Run Step 2 again.')
    _clean_records = [{col: row[col] for col in _clean_columns} for row in _reader]

# Catch accidental changes to the FormSG table layout before exporting.
_patterns = {
    'No.4 Shirt (Size, Qty)': r'No\.4 Shirt size (?:[1-9]|10);\d+',
    'No.4 Trousers (Size, Qty)': r'No\.4 Trouser size (?:[1-9]|1[0-4]);\d+',
    'PCG No.4 Utility Belt (Size, Qty)': r'Utility Belt size (?:S|M|L|XL);\d+',
    'PCG No.4 Tactical Belt (Size, Qty)': r'Tactical Belt size (?:S|M|L|XL);\d+',
    'Fire & Cut Resistant Tactical Gloves (Size, Qty)': r'Fire & Cut Resistant Tactical Gloves Size (?:S|M|L|XL);\d+'
}
for _n, _record in enumerate(_clean_records, start=2):
    if not _record['Intake'].isdigit():
        raise ValueError(f'Invalid intake number at data row {_n}.')
    try:
        datetime.strptime(_record['Timestamp'], '%d/%m/%Y %H:%M')
    except ValueError:
        raise ValueError(f'Invalid timestamp at data row {_n}.') from None
    for _column, _pattern in _patterns.items():
        if not re.fullmatch(_pattern, _record[_column]):
            raise ValueError(f'Unrecognised uniform size or quantity at data row {_n}, column {_column}.')
    for _column in _clean_columns[7:]:
        if not _record[_column].isdigit():
            raise ValueError(f'Invalid item quantity at data row {_n}, column {_column}.')

# Write and verify locally before copying the completed file back to Drive.
_clean_local = _local_folder / 'Cleaned Data.csv'
with _clean_local.open('w', encoding='utf-8-sig', newline='') as file:
    writer = csv.DictWriter(file, fieldnames=_clean_columns)
    writer.writeheader()
    writer.writerows(_clean_records)

_clean_drive = _drive_folder / 'Cleaned Data.csv'
shutil.copyfile(_clean_local, _clean_drive)

print('Step 3 complete. Cleaned Data.csv has been saved to My Drive.')
print(f'{len(_clean_records):,} responses and 9 uniform-data columns are ready.')
print('Please run Step 4 to check the results.')
# Remove local working files after the completed export.
_working_path.unlink(missing_ok=True)
_clean_local.unlink(missing_ok=True)

# Clear temporary notebook variables left by earlier versions.
for variable in (
    '_raw_lines',
    '_raw_columns',
    '_working_records',
    '_response',
    '_values'
):
    globals().pop(variable, None)

print('Temporary working files removed.')
print('Cleaned Data.csv is ready.')

In [ ]:
# @title 4. Check the cleaned data { display-mode: "form", single-column: true }
# @markdown Click **▶ Run** to see a sample of the cleaned file and check the number of personnel in each intake.
import pandas as pd
from IPython.display import display

if '_clean_drive' not in globals() or not _clean_drive.is_file():
    raise RuntimeError('Please run Step 3 first.')

_clean_preview = pd.read_csv(_clean_drive, encoding='utf-8-sig')
print('Data preparation completed successfully.')
print(f'File saved in /content/oracle_engine_sanitizer: {_clean_drive.name}')
print(f'Total responses: {len(_clean_preview):,}')
print(f'Columns retained: {len(_clean_preview.columns)}')
print('\nFirst 10 records:')
display(_clean_preview.head(10))
print('\nNumber of people in each intake:')
display(_clean_preview['Intake'].value_counts().sort_index().rename('People').to_frame())